# 00 Data Processing


In [9]:
%load_ext autoreload
%autoreload 2

import pickle
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from numpy import linalg as la

from config import (
    DATA_DIR,
    MOLECULE_DIR,
    PARAMETERS_DIR,
    RAW_MATRICES_DIR,
    PROCESSED_DATAFRAMES_DIR
)
from notebook_utils.general import complex_matrix, read_jsonl

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Statevector Dataset

In [18]:
SV_HAMILTONIAN_DIR = RAW_MATRICES_DIR / "SV_hamiltonian"
SV_SPIN_DIR = RAW_MATRICES_DIR / "SV_spin"
PYSCF_CASCI_ENERGIES_PATH = RAW_MATRICES_DIR / "pyscf_casci_energies.jsonl"
def load_sv_hamiltonian_records(path):
    records = []

    for file in sorted(path.glob("*.jsonl")):
        for row in read_jsonl(file):
            H = complex_matrix(row["h_matrix_real"], row["h_matrix_imag"])
            S = complex_matrix(row["s_matrix_real"], row["s_matrix_imag"])

            records.append({
                "molecule": row["molecule"],
                "active_space": row["active_space"],
                "ansatz": row["ansatz"],
                "expansion": row["expansion"],
                "H": H,
                "S": S,
                "qse_dim": H.shape[0],
            })

    return pd.DataFrame(records)


def load_spin_records(path):
    records = []

    for file in sorted(path.glob("*.jsonl")):
        for row in read_jsonl(file):
            S2 = complex_matrix(row["z_matrix_real"], row["z_matrix_imag"])

            records.append({
                "molecule": row["molecule"],
                "active_space": row["active_space"],
                "ansatz": row["ansatz"],
                "expansion": row["expansion"],
                "S2": S2
            })

    return pd.DataFrame(records)

def load_pyscf_casci_energy_records(path):
    records = []

    for row in read_jsonl(path):
        records.append({
            "molecule": row["molecule"],
            "active_space": row["active_space"],
            "spin_type": row["spin_type"],
            "pyscf_casci_energies": row["casci_energies"],
            "pyscf_casci_pvec": row["casci_pvec"],
        })

    return pd.DataFrame(records)


df_sv_hamiltonian = load_sv_hamiltonian_records(SV_HAMILTONIAN_DIR)
df_sv_spin = load_spin_records(SV_SPIN_DIR)
df_pyscf_casci = load_pyscf_casci_energy_records(PYSCF_CASCI_ENERGIES_PATH)


In [19]:
# Compress the PYSCF Triplet energies
def average_triplet_sector_energies(casci_energies):
    average_energies = []
    sector_diffs = []

    for nth_energies in casci_energies:
        energies = list(nth_energies.values())
        average_energies.append(float(np.mean(energies)))
        sector_diffs.append(float(np.ptp(energies)))

    return average_energies, sector_diffs


new_casci_energies = []
triplet_sector_diffs = []

for _, row in df_pyscf_casci.iterrows():
    if row["spin_type"] == "triplet_all":
        energies, sector_diffs = average_triplet_sector_energies(
            row["pyscf_casci_energies"]
        )
        triplet_sector_diffs.extend(sector_diffs)
    else:
        energies = row["pyscf_casci_energies"]

    new_casci_energies.append(energies)


df_pyscf_casci["pyscf_casci_energies"] = new_casci_energies

max_triplet_sector_diff = max(triplet_sector_diffs, default=0.0)
print(f"Maximum PYSCF triplet sector energy difference: {max_triplet_sector_diff:.12e} Ha")


Maximum PYSCF triplet sector energy difference: 1.003218130791e-08 Ha


In [20]:
df_sv = df_sv_hamiltonian.merge(
    df_sv_spin,
    on=["molecule", "active_space", "ansatz", "expansion"],
    how="left",
)

df_sv = df_sv.merge(
    df_pyscf_casci,
    left_on=["molecule", "active_space", "expansion"],
    right_on=["molecule", "active_space", "spin_type"],
    how="left",
).drop(columns="spin_type")

df_sv = df_sv.sort_values(
    ["molecule", "active_space", "ansatz", "expansion"]
).reset_index(drop=True)

In [21]:
required_columns = ["H", "S", "S2", "pyscf_casci_energies", "pyscf_casci_pvec"]

df_sv_complete = df_sv.dropna(subset=required_columns).reset_index(drop=True)

PROCESSED_DATAFRAMES_DIR.mkdir(parents=True, exist_ok=True)
SV_DATAFRAME_PATH = PROCESSED_DATAFRAMES_DIR / "sv_qse_data.pkl"

df_sv_complete.to_pickle(SV_DATAFRAME_PATH)

df_sv_complete


,molecule,active_space,ansatz,expansion,H,S,qse_dim,S2,pyscf_casci_energies,pyscf_casci_pvec
0,Acetamide,2e2o,1UpCCGSDSinglet,singlet,"[[(-802.5663414572622+0j), (-4.706692634862637...","[[(3.9097041888243123+0j), (0.0229245233142809...",4,"[[(1.27675647831893e-14+0j), (-4.3368086899420...","[-205.29449911826808, -204.8258180441668, -204...","{'0': [-0.15005494805049122, 0, -0.00628487098..."
1,Acetamide,2e2o,1UpCCGSDSinglet,triplet_all,"[[(-0.008069504761204022+0j), 0j, 0j, (0.19305...","[[(3.934912814912428e-05+0j), 0j, 0j, (-0.0009...",12,"[[(7.869825629530647e-05+0j), 0j, 0j, (-0.0018...",[-205.07455054622508],"{'0_0': [-4.001328838134542e-16, 0, -0.7071067..."
2,Acetamide,2e2o,UCCGSD,singlet,"[[(-802.5658955566977+0j), (-4.714686911530124...","[[(3.909702017207626+0j), (0.02296346038353209...",4,"[[(1.2961853812498703e-14+0j), (-4.33680868994...","[-205.29449911826808, -204.8258180441668, -204...","{'0': [-0.15005494805049122, 0, -0.00628487098..."
3,Acetamide,2e2o,UCCGSD,triplet_all,"[[(-0.008087395184562356+0j), 0j, 0j, (0.19324...","[[(3.943636678180318e-05+0j), 0j, 0j, (-0.0009...",12,"[[(7.887273356062263e-05+0j), 0j, 0j, (-0.0018...",[-205.07455054622508],"{'0_0': [-4.001328838134542e-16, 0, -0.7071067..."
4,Acetamide,2e2o,UCCSD,singlet,"[[(-802.5658955566977+0j), (-4.714686911530124...","[[(3.909702017207626+0j), (0.02296346038353209...",4,"[[(1.2961853812498703e-14+0j), (-4.33680868994...","[-205.29449911826808, -204.8258180441668, -204...","{'0': [-0.15005494805049122, 0, -0.00628487098..."
...,...,...,...,...,...,...,...,...,...,...
1339,Uracil,6e5o,UCCGSD,triplet_all,"[[(-1.2856560260843253e-10+0j), 0j, 0j, (-5.98...","[[(3.167743845011728e-13+0j), 0j, 0j, (1.47389...",75,"[[(7.19757586864489e-13+0j), 0j, 0j, 0j, 0j, 0...","[-406.93960384512866, -406.85734583999374, -40...","{'0_0': [2.5283724731332295e-16, 0, -2.7969232..."
1340,Uracil,6e5o,UCCSD,singlet,"[[(-1628.2616143008518+0j), (-3.83856517592802...","[[(3.999599413196687+0j), (9.429417583415311e-...",25,"[[(3.2520084999187885e-07+0j), 0j, 0j, 0j, 0j,...","[-407.10626545059665, -406.84826030871534, -40...","{'0': [-0.0006447180846954902, 0, 0.0003248312..."
1341,Uracil,6e5o,UCCSD,triplet_all,"[[(-4.588684987538727e-11+0j), 0j, 0j, (-5.739...","[[(1.1304845948245656e-13+0j), 0j, 0j, (1.4125...",75,"[[(2.545741395465484e-13+0j), 0j, 0j, 0j, 0j, ...","[-406.93960384512866, -406.85734583999374, -40...","{'0_0': [2.5283724731332295e-16, 0, -2.7969232..."
1342,Uracil,6e5o,UCCSDSinglet,singlet,"[[(-1628.2602220719214+0j), (-3.84238717048790...","[[(3.9995959933152356+0j), (9.438814510607852e...",25,"[[(2.048778721665213e-07+0j), 0j, 0j, 0j, 0j, ...","[-407.10626545059665, -406.84826030871534, -40...","{'0': [-0.0006447180846954902, 0, 0.0003248312..."


### Shots dataset

In [16]:
SHOTS_HAMILTONIAN_DIR = RAW_MATRICES_DIR / "shots_hamiltonian"


def load_shots_hamiltonian_records(path):
    records = []

    for file in sorted(path.glob("*.jsonl")):
        for row in read_jsonl(file):
            H = complex_matrix(row["h_matrix_real"], row["h_matrix_imag"])
            S = complex_matrix(row["s_matrix_real"], row["s_matrix_imag"])

            records.append({
                "molecule": row["molecule"],
                "active_space": row["active_space"],
                "ansatz": row["ansatz"],
                "expansion": row["expansion"],
                "sample_key": row["sample_key"],
                "n_shots": row["n_shots"],
                "repeat": row["repeat"],
                "H_shots": H,
                "S_shots": S,
                "qse_dim": H.shape[0],
            })

    return pd.DataFrame(records)


df_shots_hamiltonian = load_shots_hamiltonian_records(SHOTS_HAMILTONIAN_DIR)

df_shots_sv = df_sv_hamiltonian.rename(columns={
    "H": "H_sv",
    "S": "S_sv",
    "qse_dim": "qse_dim_sv",
})

df_shots = df_shots_hamiltonian.merge(
    df_shots_sv,
    on=["molecule", "active_space", "ansatz", "expansion"],
    how="left",
)

df_shots = df_shots.sort_values(
    ["molecule", "active_space", "ansatz", "expansion", "n_shots", "repeat"]
).reset_index(drop=True)


In [17]:
required_columns = ["H_shots", "S_shots", "H_sv", "S_sv"]

df_shots_complete = df_shots.dropna(subset=required_columns).reset_index(drop=True)

PROCESSED_DATAFRAMES_DIR.mkdir(parents=True, exist_ok=True)
SHOTS_DATAFRAME_PATH = PROCESSED_DATAFRAMES_DIR / "shots_qse_data.pkl"

df_shots_complete.to_pickle(SHOTS_DATAFRAME_PATH)

df_shots_complete


,molecule,active_space,ansatz,expansion,sample_key,n_shots,repeat,H_shots,S_shots,qse_dim,H_sv,S_sv,qse_dim_sv
0,Acetamide,2e2o,UCCSD,singlet,shots_1000_0,1000,0,"[[(-799.5517868114334+0j), (3.8471454421130837...","[[(3.8949999999999996+0j), (-0.018749999999999...",4,"[[(-802.5658955566977+0j), (-4.714686911530124...","[[(3.909702017207626+0j), (0.02296346038353209...",4
1,Acetamide,2e2o,UCCSD,singlet,shots_1000_1,1000,1,"[[(-799.3450014736095+0j), (-18.06551591062977...","[[(3.894+0j), (0.08800000000000004-0.020750000...",4,"[[(-802.5658955566977+0j), (-4.714686911530124...","[[(3.909702017207626+0j), (0.02296346038353209...",4
2,Acetamide,2e2o,UCCSD,singlet,shots_1000_2,1000,2,"[[(-802.215783891827+0j), (5.748233963083942+1...","[[(3.9080000000000004+0j), (-0.028000000000000...",4,"[[(-802.5658955566977+0j), (-4.714686911530124...","[[(3.909702017207626+0j), (0.02296346038353209...",4
3,Acetamide,2e2o,UCCSD,singlet,shots_1000_3,1000,3,"[[(-800.1703772137878+0j), (-2.411631973619889...","[[(3.898+0j), (0.01175000000000001+0.026249999...",4,"[[(-802.5658955566977+0j), (-4.714686911530124...","[[(3.909702017207626+0j), (0.02296346038353209...",4
4,Acetamide,2e2o,UCCSD,singlet,shots_1000_4,1000,4,"[[(-801.3992815429581+0j), (1.2815546057821785...","[[(3.904+0j), (-0.006250000000000016-0.0167500...",4,"[[(-802.5658955566977+0j), (-4.714686911530124...","[[(3.909702017207626+0j), (0.02296346038353209...",4
...,...,...,...,...,...,...,...,...,...,...,...,...,...
44795,Uracil,4e4o,UCCSDSinglet,triplet_all,shots_1000000_5,1000000,5,"[[(-2.82046153998715+0j), (0.02769062833778579...","[[(0.006938+0j), (-6.82358043844689e-05+4.6669...",48,"[[(-2.8105937605251006+0j), 0j, 0j, (-0.424226...","[[(0.006913608066414251+0j), 0j, 0j, (0.001044...",48
44796,Uracil,4e4o,UCCSDSinglet,triplet_all,shots_1000000_6,1000000,6,"[[(-2.7811017095414456+0j), (0.075534954229300...","[[(0.0068410000000000415+0j), (-0.000185969083...",48,"[[(-2.8105937605251006+0j), 0j, 0j, (-0.424226...","[[(0.006913608066414251+0j), 0j, 0j, (0.001044...",48
44797,Uracil,4e4o,UCCSDSinglet,triplet_all,shots_1000000_7,1000000,7,"[[(-2.8232902692329844+0j), (0.068577334203353...","[[(0.006944999999999979+0j), (-0.0001686449673...",48,"[[(-2.8105937605251006+0j), 0j, 0j, (-0.424226...","[[(0.006913608066414251+0j), 0j, 0j, (0.001044...",48
44798,Uracil,4e4o,UCCSDSinglet,triplet_all,shots_1000000_8,1000000,8,"[[(-2.8251684793253986+0j), (-0.00946524375246...","[[(0.006949499999999997+0j), (2.33345237791899...",48,"[[(-2.8105937605251006+0j), 0j, 0j, (-0.424226...","[[(0.006913608066414251+0j), 0j, 0j, (0.001044...",48
